#method-1 manual (not used)

In [1]:
import pandas as pd
import glob
import os

# Path where all your PDB CSV batch files are stored
path = r"/home/user/Downloads/test_pdb_recheck/*"   # change this folder path

# Get all CSV files
files = glob.glob(path)

# List to store dfs
dfs = []

for f in files:
    df = pd.read_csv(f)
    dfs.append(df)

# Concatenate all rows
final_df = pd.concat(dfs, ignore_index=True)

# OPTIONAL: drop duplicate rows if any
final_df = final_df.drop_duplicates()

# Save final merged CSV
final_df.to_csv("merged_pdb_data.csv", index=False)

print("Done! Merged file saved as merged_pdb_data.csv")


Done! Merged file saved as merged_pdb_data.csv


In [3]:
import pandas as pd

df = pd.read_csv("merged_pdb_data.csv", header=None)

# Row 0 = junk titles, Row 1 = real header
new_header = df.iloc[1]   # row 1
df = df[2:]               # keep everything after row 1
df.columns = new_header   # set proper header

df = df.reset_index(drop=True)


In [4]:
df.columns = df.columns.str.strip()


In [5]:
# Drop empty rows
df = df.dropna(how="all")

# Count ligands per PDB
counts = df["PDB ID"].value_counts()

single_ligand = counts[counts == 1].index

df_single = df[df["PDB ID"].isin(single_ligand)]

df_single.to_csv("pdb_single_ligand_only.csv", index=False)

print("Filtered complexes saved!")


Filtered complexes saved!


In [6]:
import pandas as pd

# Load CSV with broken header
df = pd.read_csv("merged_pdb_data.csv", header=None)

# Fix header
new_header = df.iloc[1]
df = df[2:]
df.columns = new_header
df = df.reset_index(drop=True)

# Clean header spaces
df.columns = df.columns.str.strip()

# FIX THE KEY PROBLEM
# Fill down missing PDB IDs
df["PDB ID"] = df["PDB ID"].ffill()

# Drop fully empty rows
df = df.dropna(how="all")

# Now count ligands per PDB
counts = df["PDB ID"].value_counts()

# Keep only those with exactly 1 ligand
single_ligand_ids = counts[counts == 1].index

df_single = df[df["PDB ID"].isin(single_ligand_ids)]

df_single.to_csv("pdb_single_ligand_only.csv", index=False)

print("Before:", df["PDB ID"].nunique())
print("After:", df_single["PDB ID"].nunique())


Before: 36675
After: 30786


In [7]:
import pandas as pd

df = pd.read_csv("merged_pdb_data.csv", header=None)

# Fix header
new_header = df.iloc[1]
df = df[2:]
df.columns = new_header
df = df.reset_index(drop=True)

# Strip column names
df.columns = df.columns.str.strip()

# Fill down missing PDB IDs (critical!)
df["PDB ID"] = df["PDB ID"].ffill()

# Remove fully blank rows
df = df.dropna(how="all")

# Count ligands per PDB ID
counts = df["PDB ID"].value_counts()

# Select PDB IDs that appear EXACTLY once
single_pdb_ids = counts[counts == 1].index

# Keep only those rows
df_single = df[df["PDB ID"].isin(single_pdb_ids)]

df_single.to_csv("pdb_single_ligand_only.csv", index=False)

print("Total PDB complexes before:", df["PDB ID"].nunique())
print("Total PDB complexes after:", df_single["PDB ID"].nunique())


Total PDB complexes before: 36675
Total PDB complexes after: 30786


In [9]:
import pandas as pd

df = pd.read_csv("pdb_single_ligand_only.csv")

# Convert Ligand ID to clean string
df["Ligand ID"] = df["Ligand ID"].astype(str).str.strip()

# Mark rows where ligand is missing
no_ligand_rows = df["Ligand ID"].isna() | (df["Ligand ID"] == "") | (df["Ligand ID"].str.lower() == "nan")

# PDB IDs that have NO ligand at all
pdb_ids_no_ligand = df.loc[no_ligand_rows, "PDB ID"].unique()

# REMOVE all rows for these PDB IDs
df_clean = df[~df["PDB ID"].isin(pdb_ids_no_ligand)]

df_clean.to_csv("pdb_single_ligand_clean.csv", index=False)

print("Removed PDB IDs (no ligand):", len(pdb_ids_no_ligand))
print("Remaining complexes:", df_clean["PDB ID"].nunique())


Removed PDB IDs (no ligand): 6385
Remaining complexes: 24401


In [22]:
# Known ions
ion_list = {
    "MG", "MN", "FE", "ZN", "NA", "K", "CA", "CU", "CO", "NI", "CD", "PT",
    "CL", "BR", "I", "F"
}

# Simple inorganic salts
inorganic_list = {
    "SO4", "PO4", "NO3", "CO3", "IOD", "IOD", "IOD", "IOD",
    "HOH", "WAT"
}

# Solvents / cryoprotectants
solvent_list = {
    "GOL", "MPD", "EDO", "DMS", "PEG", "PGE", "PG4", "ACT", "ACM",
    "IPA", "BDH", "2PE", "TRI"
}

# Buffers
buffer_list = {
    "TRS", "MES", "HEP", "EPE", "CIT", "TAR", "FLC"
}

# Detergents / glycols
detergent_list = {
    "NAG", "BOG", "NDG", "OG", "DDM", "LMG", "LMNG"
}

# Combined blacklist
blacklist = set().union(ion_list, inorganic_list, solvent_list, buffer_list, detergent_list)


In [11]:
df = pd.read_csv("pdb_single_ligand_clean.csv")

df["Ligand ID"] = df["Ligand ID"].astype(str).str.strip()

# Remove complexes with ligand ID in blacklist
df = df[~df["Ligand ID"].isin(blacklist)]


In [12]:
import re

atomic_weights = {
    "H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999,
    "P": 30.974, "S": 32.06, "F": 18.998, "Cl": 35.45,
    "Mg": 24.305, "Na": 22.99, "K": 39.098, "Ca": 40.078,
    "Zn": 65.38, "Fe": 55.845, "Br": 79.904, "I": 126.904
}

def estimate_mw(formula):
    if pd.isna(formula):
        return 0
    tokens = re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    mw = 0
    for atom, count in tokens:
        count = int(count) if count else 1
        mw += atomic_weights.get(atom, 0) * count
    return mw


In [13]:
df["MW"] = df["Ligand Formula"].apply(estimate_mw)


In [14]:
df = df[df["MW"] >= 150]


In [15]:
df.to_csv("pdb_clean_final.csv", index=False)
print("Final clean complexes:", df["PDB ID"].nunique())


Final clean complexes: 12625


In [21]:
import pandas as pd
import glob
import os

path = "/home/user/Downloads/blind_pdb_LOI/"
all_files = glob.glob(os.path.join(path, "*.csv"))

df_list = []

print("Files detected:", len(all_files))

for file in all_files:
    print("Reading:", file)
    df = pd.read_csv(file)
    df_list.append(df)

# simple concatenation
final_df = pd.concat(df_list, ignore_index=True)

# drop fully empty rows (optional)
final_df.dropna(how="all", inplace=True)

# save output
output = "/home/user/Downloads/blind_pdb_LOI/merged_pdb_data.csv"
final_df.to_csv(output, index=False)

print("Saved merged CSV at:", output)
print("Final shape:", final_df.shape)


Files detected: 15
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_07501-10000.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_17501-20000.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_32501-35000.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_30001-32500.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_22501-25000.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_12501-15000.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_05001-07500.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_custom_report_aff9e2fbb383913509e4cd82960b4da2_25001-27500.csv
Reading: /home/user/Downloads/blind_pdb_LOI/rcsb_pdb_

In [23]:
df = pd.read_csv("/home/user/Downloads/blind_pdb_LOI/loi.csv")

df["Ligand ID"] = df["Ligand ID"].astype(str).str.strip()

# Remove complexes with ligand ID in blacklist
df = df[~df["Ligand ID"].isin(blacklist)]
df.to_csv("pdb_clean_final.csv", index=False)
print("Final clean complexes:", df["PDB ID"].nunique())



Final clean complexes: 5092


In [1]:
import pandas as pd

# Paths
desc_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_clean.csv"
filtered_ids_path = "/home/user/Downloads/blind_pdb_LOI/loi.csv"

# Load files
df_desc = pd.read_csv(desc_path)
df_ids = pd.read_csv(filtered_ids_path)

# Clean column names just in case
df_desc.columns = df_desc.columns.str.strip()
df_ids.columns = df_ids.columns.str.strip()

# Extract the PDB codes from filtered file
# (column name should match your filtered file — adjust if different)
allowed_ids = df_ids["PDB ID"].astype(str).str.strip().unique()

# Filter descriptor file
df_filtered_desc = df_desc[df_desc["Complex_ID"].astype(str).str.strip().isin(allowed_ids)]

# Save result
output_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_new.csv"
df_filtered_desc.to_csv(output_path, index=False)

print("Original descriptor rows:", len(df_desc))
print("Remaining after filtering:", len(df_filtered_desc))
print("Saved to:", output_path)


Original descriptor rows: 5265
Remaining after filtering: 1667
Saved to: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/ECIF_descriptors_test_new.csv


In [3]:
import pandas as pd

# Paths
desc_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Merged_ECIF_RDKit_test_nonoverlapped.csv"
filtered_ids_path = "/home/user/Downloads/blind_pdb_LOI/loi.csv"

# Load files
df_desc = pd.read_csv(desc_path)
df_ids = pd.read_csv(filtered_ids_path)

# Clean column names just in case
df_desc.columns = df_desc.columns.str.strip()
df_ids.columns = df_ids.columns.str.strip()

# Extract the PDB codes from filtered file
# (column name should match your filtered file — adjust if different)
allowed_ids = df_ids["PDB ID"].astype(str).str.strip().unique()

# Filter descriptor file
df_filtered_desc = df_desc[df_desc["Complex_ID"].astype(str).str.strip().isin(allowed_ids)]

# Save result
output_path = "/media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/merged_descriptors_test_new.csv"
df_filtered_desc.to_csv(output_path, index=False)

print("Original descriptor rows:", len(df_desc))
print("Remaining after filtering:", len(df_filtered_desc))
print("Saved to:", output_path)


Original descriptor rows: 5265
Remaining after filtering: 1667
Saved to: /media/user/282895b1-543c-4689-9ced-b38fcef5e554/YASHI_ECIF/Trial/merged_descriptors_test_new.csv


In [6]:
import pandas as pd
import glob

path = "/home/user/Downloads/blind_pdb_LOI/*.csv"

dfs = []

for f in glob.glob(path):
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()  # clean header
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)

# Save merged file
merged.to_csv("/home/user/Downloads/blind_pdb_LOI/merged_raw.csv", index=False)
print("Merged file saved as merged_raw.csv")


Merged file saved as merged_raw.csv


In [8]:
df = pd.read_csv("/home/user/Downloads/blind_pdb_LOI/merged_raw.csv")

# Normalize PDB ID column
df["PDB ID"] = df["PDB ID"].astype(str).str.strip()

# Count ligands per PDB
counts = df["PDB ID"].value_counts()

# Keep PDBs with exactly one ligand row
one_ligand_ids = counts[counts == 1].index

df_step2 = df[df["PDB ID"].isin(one_ligand_ids)]

df_step2.to_csv("step2_single_ligand_only.csv", index=False)
print("After Step 2:", df_step2["PDB ID"].nunique(), "PDBs")


After Step 2: 36675 PDBs


In [10]:
df_step2["Ligand of Interest"] = df_step2["Ligand of Interest"].astype(str).str.strip()

df_final = df_step2[df_step2["Ligand of Interest"] == "Y"]

df_final.to_csv("/home/user/Downloads/blind_pdb_LOI/pdb_clean_LOI_single.csv", index=False)
print("Final PDB count:", df_final["PDB ID"].nunique())


Final PDB count: 6529


/tmp/ipykernel_2918563/916043949.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_step2["Ligand of Interest"] = df_step2["Ligand of Interest"].astype(str).str.strip()
